# UK Train Rides: Cleaning and Transformation

This notebook cleans the raw railway data and builds the 11 tables of the project schema (`docs/UK_Train_Rides_Schema.docx`): 9 dimension tables and 2 fact tables. The tables are saved as CSV files in `data/processed/` so they can be loaded into SQL Server.

The issues fixed here were found in `01_data_profiling.ipynb`. No rows are removed.

The notebook has five parts:

- **Part A**: Cleaning
- **Part B**: Dimension tables
- **Part C**: Fact tables
- **Part D**: Validation
- **Part E**: Export

## Part A: Cleaning

### A1. Load the data

We read the raw file and save the row count and total price. At the end we will check that both numbers did not change.

In [1]:
import pandas as pd

pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/railway.csv")

raw_rows = len(df)
raw_total_price = df["Price"].sum()
print("Rows:", raw_rows)
print("Total price: £", raw_total_price)

Rows: 31653
Total price: £ 741921


### A2. Standardize the delay reasons

Three reasons are written in two different ways. We replace the second spelling with the first one.

In [2]:
reason_fix = {
    "Signal failure": "Signal Failure",
    "Weather Conditions": "Weather",
    "Staff Shortage": "Staffing",
}
df["Reason for Delay"] = df["Reason for Delay"].replace(reason_fix)

df["Reason for Delay"].value_counts(dropna=False)

Reason for Delay
NaN                27481
Weather             1372
Signal Failure       970
Staffing             809
Technical Issue      707
Traffic              314
Name: count, dtype: int64

Now there are five reasons: Weather (1,372), Signal Failure (970), Staffing (809), Technical Issue (707) and Traffic (314). The 27,481 empty values are still there.

From the profiling notebook we know that a reason is empty only when the journey is on time. We fill these values with `No Delay`, so every journey has a value in this column. This will also be a row in the delay reason dimension table.

In [3]:
df["Reason for Delay"] = df["Reason for Delay"].fillna("No Delay")

pd.crosstab(df["Reason for Delay"], df["Journey Status"])

Journey Status,Cancelled,Delayed,On Time
Reason for Delay,,,
No Delay,0,0,27481
Signal Failure,519,451,0
Staffing,454,355,0
Technical Issue,235,472,0
Traffic,227,87,0
Weather,445,927,0


`No Delay` matches exactly the 27,481 on-time tickets, and on-time tickets have no other reason. Delayed and cancelled tickets always have a real reason.

### A3. Fill the missing railcards

The data dictionary says a passenger without a railcard is "None". We use this value for the empty cells.

In [4]:
df["Railcard"] = df["Railcard"].fillna("None")

df["Railcard"].value_counts()

Railcard
None        20918
Adult        4846
Disabled     3089
Senior       2800
Name: count, dtype: int64

The 20,918 empty values are now `None`. The other railcard counts did not change.

### A4. Convert dates and times

Dates are converted to datetime and times to time values. If a value had a wrong format, the conversion would raise an error, so this step also checks that all values are valid.

`Actual Arrival Time` stays empty for cancelled journeys.

In [5]:
date_cols = ["Date of Purchase", "Date of Journey"]
time_cols = ["Time of Purchase", "Departure Time", "Arrival Time", "Actual Arrival Time"]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], format="%Y-%m-%d")

for col in time_cols:
    df[col] = pd.to_datetime(df[col], format="%H:%M:%S").dt.time

df[date_cols + time_cols].head()

,Date of Purchase,Date of Journey,Time of Purchase,Departure Time,Arrival Time,Actual Arrival Time
0,2023-12-08,2024-01-01,12:41:11,11:00:00,13:30:00,13:30:00
1,2023-12-16,2024-01-01,11:23:01,09:45:00,11:35:00,11:40:00
2,2023-12-19,2024-01-02,19:51:27,18:15:00,18:45:00,18:45:00
3,2023-12-20,2024-01-01,23:00:36,21:30:00,22:30:00,22:30:00
4,2023-12-27,2024-01-01,18:22:56,16:45:00,19:00:00,19:00:00


In [6]:
df[date_cols + time_cols].isna().sum()

Date of Purchase          0
Date of Journey           0
Time of Purchase          0
Departure Time            0
Arrival Time              0
Actual Arrival Time    1880
dtype: int64

All dates and times were converted without errors. The only empty values are the 1,880 actual arrival times of cancelled journeys, as expected.

### A5. Days booked ahead

The number of days between the purchase date and the journey date. This becomes the `Days_Booked_Ahead` column of the ticket sales table.

In [7]:
df["Days_Booked_Ahead"] = (df["Date of Journey"] - df["Date of Purchase"]).dt.days

df["Days_Booked_Ahead"].describe().round(1)

count    31653.0
mean         1.7
std          3.9
min          0.0
25%          0.0
50%          1.0
75%          1.0
max         28.0
Name: Days_Booked_Ahead, dtype: float64

The values go from 0 to 28 days, with an average of 1.7, the same as in the profiling notebook.

### A6. Delay in minutes

Some trains arrive after midnight, so we cannot subtract clock times directly. We build full date-time values instead:

1. Departure = journey date + departure time.
2. Scheduled and actual arrival = journey date + arrival time.
3. If an arrival time is earlier than the departure time, the train arrived on the next day, so we add one day.
4. Delay = actual arrival minus scheduled arrival, in minutes.

Cancelled journeys have no actual arrival, so their delay stays empty instead of being set to 0.

In [8]:
def to_datetime_on_journey_day(time_col):
    # Combine the journey date with a time column
    return df["Date of Journey"] + pd.to_timedelta(df[time_col].astype(str))

departure_dt = to_datetime_on_journey_day("Departure Time")
scheduled_dt = to_datetime_on_journey_day("Arrival Time")
actual_dt = to_datetime_on_journey_day("Actual Arrival Time")

# Arrival earlier than departure means the train arrived the next day
one_day = pd.Timedelta(days=1)
scheduled_dt = scheduled_dt.where(scheduled_dt >= departure_dt, scheduled_dt + one_day)
actual_dt = actual_dt.where(actual_dt.isna() | (actual_dt >= departure_dt), actual_dt + one_day)

df["Delay_Minutes"] = ((actual_dt - scheduled_dt).dt.total_seconds() / 60).astype("Int64")

df.groupby("Journey Status")["Delay_Minutes"].agg(["count", "min", "max", "mean"]).round(1)

,count,min,max,mean
Journey Status,,,,
Cancelled,0,<NA>,<NA>,<NA>
Delayed,2292,0,180,42.2
On Time,27481,0,0,0.0


The result for each status:

- **On Time**: 27,481 tickets, all with 0 minutes delay.
- **Delayed**: 2,292 tickets, from 0 to 180 minutes, average 42.2. This matches the profiling notebook.
- **Cancelled**: count is 0 because the delay is empty for all of them, which is what we want.

The 18 delayed tickets with 0 minutes delay are still there. We keep them as they are, as decided in the profiling notebook.

We check that the midnight rule worked. The profiling notebook found 961 tickets with a scheduled arrival after midnight, so the same number should now have their arrival moved to the next day. We also check that no delay is negative.

In [9]:
next_day = scheduled_dt.dt.normalize() > df["Date of Journey"]
print("Scheduled arrivals moved to the next day:", next_day.sum())
print("Negative delays:", (df["Delay_Minutes"] < 0).sum())

Scheduled arrivals moved to the next day: 961
Negative delays: 0


961 arrivals were moved to the next day, which matches the profiling result, and there are no negative delays.

### A7. Check the cleaned data

Cleaning should not add or remove rows or change the prices.

In [10]:
print("Rows before:", raw_rows, "| after:", len(df))
print("Total price before: £", raw_total_price, "| after: £", df["Price"].sum())

assert len(df) == raw_rows
assert df["Price"].sum() == raw_total_price

Rows before: 31653 | after: 31653
Total price before: £ 741921 | after: £ 741921


The row count (31,653) and the total price (£741,921) are unchanged. The data is now ready to be split into dimension and fact tables.

## Part B: Dimension tables

Each dimension table holds the distinct values of one column, with a numeric ID as the primary key. The fact tables will store these IDs instead of repeating the text.

The rule for all IDs: they start at 1 and follow the alphabetical order of the values. This gives the same IDs every time the notebook runs.

We use a small helper function for the simple dimensions.

In [11]:
def make_dimension(values, id_col, name_col):
    # Distinct values sorted alphabetically, with IDs starting at 1
    names = sorted(pd.Series(values).unique())
    return pd.DataFrame({id_col: range(1, len(names) + 1), name_col: names})

### B1. Dim_Station

The stations come from both the departure and the arrival columns. The schema also needs `City` and `Region`, which are not in the CSV.

- **City**: in this dataset the first word of every station name is the city (for example `London Euston` is in London and `Cardiff Central` is in Cardiff). We take the first word and then check the result by eye.
- **Region**: we take it from an official file published by the Office of Rail and Road (ORR), the UK railway regulator: *Table 6329: Station attributes for all mainline stations* (`data/orr_station_attributes.ods`). The same file also gives the location of every station, which we use for the route distances in B2.

We start by building the station list and the city.

In [12]:
all_stations = pd.concat([df["Departure Station"], df["Arrival Destination"]])
dim_station = make_dimension(all_stations, "StationID", "Station_Name")

dim_station["City"] = dim_station["Station_Name"].str.split().str[0]
dim_station

,StationID,Station_Name,City
0,1,Birmingham New Street,Birmingham
1,2,Bristol Temple Meads,Bristol
2,3,Cardiff Central,Cardiff
3,4,Coventry,Coventry
4,5,Crewe,Crewe
5,6,Didcot,Didcot
6,7,Doncaster,Doncaster
7,8,Durham,Durham
8,9,Edinburgh,Edinburgh
9,10,Edinburgh Waverley,Edinburgh


The table has 32 stations. Checking the list by eye, the first-word rule gives the correct city for all of them.

Next we read the ORR file. The table starts at row 4 of the sheet `6329_station_attributes`, so we skip the title rows. We keep only the columns we need.

Reading `.ods` files needs the `odfpy` package (`pip install odfpy`).

In [13]:
orr = pd.read_excel("../data/orr_station_attributes.ods",
                    sheet_name="6329_station_attributes", header=3)

orr = orr[["Station name", "Region",
           "Ordnance Survey grid: Easting", "Ordnance Survey grid: Northing"]]
orr.columns = ["ORR_Name", "Region", "Easting", "Northing"]

print("Stations in the ORR file:", len(orr))
orr.head()

Stations in the ORR file: 2595


,ORR_Name,Region,Easting,Northing
0,Abbey Wood,London,547273,179032
1,Aber,Wales,314867,186951
2,Abercynon,Wales,308107,194652
3,Aberdare,Wales,300479,202743
4,Aberdeen,Scotland,394162,805794


26 of our 32 station names exist in the ORR file exactly as they are. The other 6 are written differently, so we match them by hand:

- `Didcot` is `Didcot Parkway`, the only station in Didcot.
- `Swindon` is `Swindon (Wilts)` and `London St Pancras` is `London St Pancras International`.
- `Edinburgh Waverley` is called just `Edinburgh` in the ORR file.
- `Wakefield` has two stations. We use `Wakefield Westgate`, the station on the main line.
- `Warrington` has three stations. We use `Warrington Central`. The three stations are less than 1 km apart, so the choice has almost no effect on distances.

In [14]:
orr_name = {
    "Didcot": "Didcot Parkway",
    "Swindon": "Swindon (Wilts)",
    "London St Pancras": "London St Pancras International",
    "Edinburgh Waverley": "Edinburgh",
    "Wakefield": "Wakefield Westgate",
    "Warrington": "Warrington Central",
}

# Name used in the ORR file (same name unless listed above)
dim_station["ORR_Name"] = dim_station["Station_Name"].replace(orr_name)

stations_orr = dim_station.merge(orr, on="ORR_Name", how="left")

assert stations_orr["Region"].notna().all(), "A station was not found in the ORR file"
print("Stations matched:", stations_orr["Region"].notna().sum(), "of", len(dim_station))

Stations matched: 32 of 32


All 32 stations were found in the ORR file. We add the region to `Dim_Station`.

We also add the location of each station, so the stations can be shown on a map in Power BI. Power BI maps need latitude and longitude, but the ORR file gives Ordnance Survey coordinates (Easting and Northing). We convert them with the `pyproj` package (`pip install pyproj`). `EPSG:27700` is the code of the British grid and `EPSG:4326` is the code of normal latitude and longitude.

`Latitude` and `Longitude` are not in the original schema. They are added only to support the map.

In [15]:
from pyproj import Transformer

to_lat_lon = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
longitude, latitude = to_lat_lon.transform(stations_orr["Easting"].to_numpy(),
                                           stations_orr["Northing"].to_numpy())

dim_station["Region"] = stations_orr["Region"]
dim_station["Latitude"] = latitude.round(6)
dim_station["Longitude"] = longitude.round(6)

dim_station = dim_station[["StationID", "Station_Name", "City", "Region", "Latitude", "Longitude"]]
dim_station

,StationID,Station_Name,City,Region,Latitude,Longitude
0,1,Birmingham New Street,Birmingham,West Midlands,52.477645,-1.898954
1,2,Bristol Temple Meads,Bristol,South West,51.449327,-2.580310
2,3,Cardiff Central,Cardiff,Wales,51.475619,-3.179334
3,4,Coventry,Coventry,West Midlands,52.400768,-1.513622
4,5,Crewe,Crewe,North West,53.089210,-2.432875
5,6,Didcot,Didcot,South East,51.611193,-1.242514
6,7,Doncaster,Doncaster,Yorkshire and The Humber,53.522148,-1.139845
7,8,Durham,Durham,North East,54.779694,-1.581609
8,9,Edinburgh,Edinburgh,Scotland,55.952115,-3.188956
9,10,Edinburgh Waverley,Edinburgh,Scotland,55.952115,-3.188956


To check the conversion, all stations should be inside Great Britain: latitude between about 50 and 59, and longitude between about -6 and 2.

In [16]:
print("Latitude: ", dim_station["Latitude"].min(), "to", dim_station["Latitude"].max())
print("Longitude:", dim_station["Longitude"].min(), "to", dim_station["Longitude"].max())

assert dim_station["Latitude"].between(50, 59).all()
assert dim_station["Longitude"].between(-6, 2).all()

Latitude:  51.449327 to 55.952115
Longitude: -3.188956 to -0.113437


The number of stations in each region:

In [17]:
dim_station["Region"].value_counts()

Region
West Midlands               6
Yorkshire and The Humber    5
London                      5
North West                  4
South East                  3
South West                  2
Scotland                    2
East Midlands               2
Wales                       1
North East                  1
East of England             1
Name: count, dtype: int64

The stations cover 11 regions. West Midlands has the most stations (6), followed by London and Yorkshire and The Humber (5 each).

### B2. Dim_Route

A route is a pair of departure and arrival stations. We replace the station names with their IDs from `Dim_Station`.

**Distance_km**: the dataset has no distances, so we calculate them from the station locations in the ORR file. The locations are Ordnance Survey grid coordinates (`Easting` and `Northing`), measured in metres on a flat map of Great Britain. This means the distance between two stations is a simple Pythagoras calculation:

distance = √((Easting₁ − Easting₂)² + (Northing₁ − Northing₂)²)

This is the **straight-line distance**. The real track distance is longer, because railway lines are not straight. We use it as an approximate measure of how long a route is.

In [18]:
station_id = dict(zip(dim_station["Station_Name"], dim_station["StationID"]))

routes = (df[["Departure Station", "Arrival Destination"]]
          .drop_duplicates()
          .sort_values(["Departure Station", "Arrival Destination"])
          .reset_index(drop=True))

# Station coordinates from the ORR file, indexed by station ID
coords = stations_orr.set_index("StationID")[["Easting", "Northing"]]

dim_route = pd.DataFrame({
    "RouteID": range(1, len(routes) + 1),
    "Departure_StationID": routes["Departure Station"].map(station_id),
    "Arrival_StationID": routes["Arrival Destination"].map(station_id),
})

dep = coords.loc[dim_route["Departure_StationID"]].to_numpy()
arr = coords.loc[dim_route["Arrival_StationID"]].to_numpy()
distance_m = ((dep - arr) ** 2).sum(axis=1) ** 0.5
dim_route["Distance_km"] = (distance_m / 1000).round(1)

assert dim_route["Distance_km"].notna().all()
print("Routes:", len(dim_route))
dim_route.head()

Routes: 65


,RouteID,Departure_StationID,Arrival_StationID,Distance_km
0,1,1,4,27.6
1,2,1,9,395.6
2,3,1,13,126.3
3,4,1,14,160.6
4,5,1,15,161.0


There are 65 routes, the same number found in the profiling notebook, and every route has a distance. Route 1 goes from station 1 (Birmingham New Street) to station 4 (Coventry).

To check that the distances make sense, we look at some well-known routes with the station names.

In [19]:
station_name = dict(zip(dim_station["StationID"], dim_station["Station_Name"]))

route_check = dim_route.assign(
    From=dim_route["Departure_StationID"].map(station_name),
    To=dim_route["Arrival_StationID"].map(station_name),
)[["RouteID", "From", "To", "Distance_km"]]

route_check.sort_values("Distance_km")

,RouteID,From,To,Distance_km
32,33,London Paddington,London Waterloo,4.7
12,13,Birmingham New Street,Wolverhampton,19.4
11,12,Birmingham New Street,Tamworth,22.9
51,52,Reading,Didcot,25.2
47,48,Manchester Piccadilly,Warrington,25.9
...,...,...,...,...
21,22,Liverpool Lime Street,London St Pancras,284.7
29,30,London Kings Cross,Liverpool Lime Street,284.8
1,2,Birmingham New Street,Edinburgh,395.6
28,29,London Kings Cross,Edinburgh Waverley,531.6


The shortest route is London Paddington to London Waterloo (4.7 km), two stations inside London. The longest is London Kings Cross to Edinburgh Waverley (531.6 km) in both directions. Well-known distances such as London Euston to Birmingham New Street (160.6 km) and Liverpool to Manchester look reasonable for straight lines.

### B3. Dim_Date

One row per calendar day. The table must cover both purchase dates (from 8 December 2023) and journey dates (until 30 April 2024). We start on 1 December 2023 so that every month is complete.

`DateKey` is the date written as a number in the format YYYYMMDD, for example 20240115 for 15 January 2024. `Is_Weekend` is 1 for Saturday and Sunday and 0 for other days.

In [20]:
dates = pd.date_range("2023-12-01", "2024-04-30", freq="D")

dim_date = pd.DataFrame({
    "DateKey": dates.strftime("%Y%m%d").astype(int),
    "Date": dates,
    "Day_Name": dates.day_name(),
    "Is_Weekend": (dates.dayofweek >= 5).astype(int),
    "Month": dates.month,
    "Quarter": dates.quarter,
    "Year": dates.year,
})

print("Days:", len(dim_date))
dim_date.head()

Days: 152


,DateKey,Date,Day_Name,Is_Weekend,Month,Quarter,Year
0,20231201,2023-12-01,Friday,0,12,4,2023
1,20231202,2023-12-02,Saturday,1,12,4,2023
2,20231203,2023-12-03,Sunday,1,12,4,2023
3,20231204,2023-12-04,Monday,0,12,4,2023
4,20231205,2023-12-05,Tuesday,0,12,4,2023


We check that every purchase date and journey date in the data exists in the date table.

In [21]:
all_dates = pd.concat([df["Date of Purchase"], df["Date of Journey"]])
print("Dates missing from Dim_Date:", (~all_dates.isin(dim_date["Date"])).sum())

Dates missing from Dim_Date: 0


The table has 152 days and covers all dates in the data.

### B4. Ticket type, ticket class and railcard

`Discount_Pct` comes from the data dictionary: Advance tickets are 1/2 off, Off-Peak 1/4 off and Anytime full price. Railcard holders get 1/3 off.

These percentages are for reference only. The `Price` column is already the final price, so the discounts are not applied again.

In [22]:
dim_ticket_type = make_dimension(df["Ticket Type"], "TicketTypeID", "Ticket_Type")
dim_ticket_type["Discount_Pct"] = dim_ticket_type["Ticket_Type"].map(
    {"Advance": 50.0, "Off-Peak": 25.0, "Anytime": 0.0})
dim_ticket_type

,TicketTypeID,Ticket_Type,Discount_Pct
0,1,Advance,50.0
1,2,Anytime,0.0
2,3,Off-Peak,25.0


In [23]:
dim_ticket_class = make_dimension(df["Ticket Class"], "TicketClassID", "Ticket_Class")
dim_ticket_class

,TicketClassID,Ticket_Class
0,1,First Class
1,2,Standard


In [24]:
dim_railcard = make_dimension(df["Railcard"], "RailcardID", "Railcard")
dim_railcard["Discount_Pct"] = [0.0 if r == "None" else 33.33 for r in dim_railcard["Railcard"]]
dim_railcard

,RailcardID,Railcard,Discount_Pct
0,1,Adult,33.33
1,2,Disabled,33.33
2,3,None,0.00
3,4,Senior,33.33


Three railcard types with a 33.33% discount, plus `None` for passengers without a railcard.

### B5. Payment method and purchase type

In [25]:
dim_payment_method = make_dimension(df["Payment Method"], "PaymentMethodID", "Payment_Method")
dim_payment_method

,PaymentMethodID,Payment_Method
0,1,Contactless
1,2,Credit Card
2,3,Debit Card


In [26]:
dim_purchase_type = make_dimension(df["Purchase Type"], "PurchaseTypeID", "Purchase_Type")
dim_purchase_type

,PurchaseTypeID,Purchase_Type
0,1,Online
1,2,Station


### B6. Dim_DelayReason

The schema asks for a `Reason_Category` column. We group the reasons by who or what caused the problem:

- **Infrastructure**: problems with the railway equipment (Signal Failure, Technical Issue).
- **Staff**: not enough staff (Staffing).
- **External**: causes outside the railway's control (Weather, Traffic).
- **No Delay**: journeys that were on time.

In [27]:
dim_delay_reason = make_dimension(df["Reason for Delay"], "DelayReasonID", "Reason_for_Delay")

reason_category = {
    "Signal Failure": "Infrastructure",
    "Technical Issue": "Infrastructure",
    "Staffing": "Staff",
    "Weather": "External",
    "Traffic": "External",
    "No Delay": "No Delay",
}
dim_delay_reason["Reason_Category"] = dim_delay_reason["Reason_for_Delay"].map(reason_category)

assert dim_delay_reason["Reason_Category"].notna().all()
dim_delay_reason

,DelayReasonID,Reason_for_Delay,Reason_Category
0,1,No Delay,No Delay
1,2,Signal Failure,Infrastructure
2,3,Staffing,Staff
3,4,Technical Issue,Infrastructure
4,5,Traffic,External
5,6,Weather,External


### B7. Summary of the dimension tables

In [28]:
dimensions = {
    "Dim_Station": dim_station,
    "Dim_Route": dim_route,
    "Dim_Date": dim_date,
    "Dim_TicketType": dim_ticket_type,
    "Dim_TicketClass": dim_ticket_class,
    "Dim_Railcard": dim_railcard,
    "Dim_PaymentMethod": dim_payment_method,
    "Dim_PurchaseType": dim_purchase_type,
    "Dim_DelayReason": dim_delay_reason,
}

for name, table in dimensions.items():
    print(f"{name:<18} {len(table):>4} rows")

Dim_Station          32 rows
Dim_Route            65 rows
Dim_Date            152 rows
Dim_TicketType        3 rows
Dim_TicketClass       2 rows
Dim_Railcard          4 rows
Dim_PaymentMethod     3 rows
Dim_PurchaseType      2 rows
Dim_DelayReason       6 rows


All 9 dimension tables are ready. Next we build the two fact tables using these IDs.

## Part C: Fact tables

### C1. Fact_Journey

One row per journey. As defined in the schema, a journey is a unique combination of route, journey date and departure time.

In the profiling notebook all tickets of the same journey had the same journey details, except one journey with a spelling difference in the delay reason. That was fixed in A2, so we confirm again before grouping.

In [29]:
journey_key = ["Departure Station", "Arrival Destination", "Date of Journey", "Departure Time"]
journey_cols = ["Arrival Time", "Actual Arrival Time", "Journey Status", "Reason for Delay", "Delay_Minutes"]

values_per_journey = df.groupby(journey_key)[journey_cols].nunique(dropna=False)
print("Journeys with conflicting values:", (values_per_journey > 1).any(axis=1).sum())

Journeys with conflicting values: 0


There are no conflicts, so we can safely keep one row per journey. We take the journey details from the first ticket of each journey.

The journeys are sorted by date, departure time and stations, and numbered from 1. Then we replace the route, date and delay reason with their IDs.

In [30]:
journeys = (df.groupby(journey_key, as_index=False)[journey_cols].first()
              .sort_values(["Date of Journey", "Departure Time", "Departure Station", "Arrival Destination"])
              .reset_index(drop=True))
journeys.insert(0, "JourneyID", range(1, len(journeys) + 1))

route_id = {
    (station_name[d], station_name[a]): r
    for r, d, a in zip(dim_route["RouteID"], dim_route["Departure_StationID"], dim_route["Arrival_StationID"])
}
delay_reason_id = dict(zip(dim_delay_reason["Reason_for_Delay"], dim_delay_reason["DelayReasonID"]))

fact_journey = pd.DataFrame({
    "JourneyID": journeys["JourneyID"],
    "RouteID": [route_id[(d, a)] for d, a in zip(journeys["Departure Station"], journeys["Arrival Destination"])],
    "JourneyDateKey": journeys["Date of Journey"].dt.strftime("%Y%m%d").astype(int),
    "DelayReasonID": journeys["Reason for Delay"].map(delay_reason_id),
    "Departure_Time": journeys["Departure Time"],
    "Scheduled_Arrival_Time": journeys["Arrival Time"],
    "Actual_Arrival_Time": journeys["Actual Arrival Time"],
    "Journey_Status": journeys["Journey Status"],
    "Delay_Minutes": journeys["Delay_Minutes"],
})

print("Journeys:", len(fact_journey))
fact_journey.head()

Journeys: 19871


,JourneyID,RouteID,JourneyDateKey,DelayReasonID,Departure_Time,Scheduled_Arrival_Time,Actual_Arrival_Time,Journey_Status,Delay_Minutes
0,1,23,20240101,1,00:00:00,00:30:00,00:30:00,On Time,0
1,2,28,20240101,2,00:00:00,01:50:00,02:07:00,Delayed,17
2,3,36,20240101,1,01:30:00,02:30:00,02:30:00,On Time,0
3,4,59,20240101,1,01:45:00,02:35:00,02:35:00,On Time,0
4,5,36,20240101,1,02:15:00,03:15:00,03:15:00,On Time,0


The table has 19,871 journeys, the same number found in the profiling notebook.

Now we can see the journey status counted the right way, once per journey, and compare it with the count per ticket.

In [31]:
status = pd.DataFrame({
    "Journeys": fact_journey["Journey_Status"].value_counts(),
    "Tickets": df["Journey Status"].value_counts(),
})
status["Journeys %"] = (status["Journeys"] / status["Journeys"].sum() * 100).round(1)
status["Tickets %"] = (status["Tickets"] / status["Tickets"].sum() * 100).round(1)
status

,Journeys,Tickets,Journeys %,Tickets %
On Time,18019,27481,90.7,86.8
Delayed,1062,2292,5.3,7.2
Cancelled,790,1880,4.0,5.9


The percentages are different depending on what we count. Per journey, 4.0% of journeys were cancelled and 5.3% were delayed. Per ticket, the same numbers look higher (5.9% and 7.2%), because delayed and cancelled journeys have more tickets on average.

This is why operational measures such as the cancellation rate must be calculated from `Fact_Journey`, not from the ticket table.

### C2. Fact_TicketSales

One row per ticket. Each ticket gets the `JourneyID` of its journey, found by matching the same four journey columns. The other text columns are replaced with the IDs from the dimension tables.

In [32]:
journey_lookup = journeys[journey_key + ["JourneyID"]]
tickets = df.merge(journey_lookup, on=journey_key, how="left", validate="many_to_one")

assert len(tickets) == len(df), "The merge changed the number of rows"
assert tickets["JourneyID"].notna().all(), "A ticket has no journey"

def lookup(dim, name_col, id_col):
    return dict(zip(dim[name_col], dim[id_col]))

fact_ticket_sales = pd.DataFrame({
    "Transaction_ID": tickets["Transaction ID"],
    "JourneyID": tickets["JourneyID"],
    "PurchaseDateKey": tickets["Date of Purchase"].dt.strftime("%Y%m%d").astype(int),
    "PurchaseTypeID": tickets["Purchase Type"].map(lookup(dim_purchase_type, "Purchase_Type", "PurchaseTypeID")),
    "PaymentMethodID": tickets["Payment Method"].map(lookup(dim_payment_method, "Payment_Method", "PaymentMethodID")),
    "RailcardID": tickets["Railcard"].map(lookup(dim_railcard, "Railcard", "RailcardID")),
    "TicketClassID": tickets["Ticket Class"].map(lookup(dim_ticket_class, "Ticket_Class", "TicketClassID")),
    "TicketTypeID": tickets["Ticket Type"].map(lookup(dim_ticket_type, "Ticket_Type", "TicketTypeID")),
    "Time_of_Purchase": tickets["Time of Purchase"],
    "Price": tickets["Price"],
    "Refund_Request": tickets["Refund Request"],
    "Days_Booked_Ahead": tickets["Days_Booked_Ahead"],
})

print("Tickets:", len(fact_ticket_sales))
fact_ticket_sales.head()

Tickets: 31653


,Transaction_ID,JourneyID,PurchaseDateKey,PurchaseTypeID,PaymentMethodID,RailcardID,TicketClassID,TicketTypeID,Time_of_Purchase,Price,Refund_Request,Days_Booked_Ahead
0,da8a6ba8-b3dc-4677-b176,34,20231208,1,1,1,2,1,12:41:11,43,No,24
1,b0cdd1b0-f214-4197-be53,32,20231216,2,2,1,2,1,11:23:01,23,No,16
2,f3ba7a96-f713-40d9-9629,152,20231219,1,2,3,2,1,19:51:27,3,No,14
3,b2471f11-4fe7-4c87-8ab4,59,20231220,2,2,3,2,1,23:00:36,13,No,12
4,2be00b45-0762-485e-a7a3,48,20231227,1,1,3,2,1,18:22:56,76,No,5


The table has 31,653 tickets, one for each row of the original data. Every ticket found its journey.

## Part D: Validation

Before exporting, we run a set of checks on all 11 tables. Each check prints PASS or FAIL.

In [33]:
tables = {**dimensions, "Fact_Journey": fact_journey, "Fact_TicketSales": fact_ticket_sales}

primary_keys = {
    "Dim_Station": "StationID", "Dim_Route": "RouteID", "Dim_Date": "DateKey",
    "Dim_TicketType": "TicketTypeID", "Dim_TicketClass": "TicketClassID",
    "Dim_Railcard": "RailcardID", "Dim_PaymentMethod": "PaymentMethodID",
    "Dim_PurchaseType": "PurchaseTypeID", "Dim_DelayReason": "DelayReasonID",
    "Fact_Journey": "JourneyID", "Fact_TicketSales": "Transaction_ID",
}

# (table, foreign key column, referenced table, referenced column)
foreign_keys = [
    ("Dim_Route", "Departure_StationID", "Dim_Station", "StationID"),
    ("Dim_Route", "Arrival_StationID", "Dim_Station", "StationID"),
    ("Fact_Journey", "RouteID", "Dim_Route", "RouteID"),
    ("Fact_Journey", "JourneyDateKey", "Dim_Date", "DateKey"),
    ("Fact_Journey", "DelayReasonID", "Dim_DelayReason", "DelayReasonID"),
    ("Fact_TicketSales", "JourneyID", "Fact_Journey", "JourneyID"),
    ("Fact_TicketSales", "PurchaseDateKey", "Dim_Date", "DateKey"),
    ("Fact_TicketSales", "PurchaseTypeID", "Dim_PurchaseType", "PurchaseTypeID"),
    ("Fact_TicketSales", "PaymentMethodID", "Dim_PaymentMethod", "PaymentMethodID"),
    ("Fact_TicketSales", "RailcardID", "Dim_Railcard", "RailcardID"),
    ("Fact_TicketSales", "TicketClassID", "Dim_TicketClass", "TicketClassID"),
    ("Fact_TicketSales", "TicketTypeID", "Dim_TicketType", "TicketTypeID"),
]

checks = []

for name, pk in primary_keys.items():
    pk_values = tables[name][pk]
    ok = pk_values.notna().all() and pk_values.is_unique
    checks.append((f"{name}.{pk} is unique and not empty", ok))

for table, fk, ref_table, ref_col in foreign_keys:
    ok = tables[table][fk].isin(tables[ref_table][ref_col]).all()
    checks.append((f"{table}.{fk} -> {ref_table}.{ref_col}", ok))

checks.append(("Fact_Journey has 19,871 rows", len(fact_journey) == 19871))
checks.append(("Fact_TicketSales has the same rows as the raw data", len(fact_ticket_sales) == raw_rows))
checks.append(("Total price is unchanged", fact_ticket_sales["Price"].sum() == raw_total_price))

for check, ok in checks:
    print("PASS" if ok else "FAIL", "|", check)

assert all(ok for _, ok in checks), "Some checks failed"

PASS | Dim_Station.StationID is unique and not empty
PASS | Dim_Route.RouteID is unique and not empty
PASS | Dim_Date.DateKey is unique and not empty
PASS | Dim_TicketType.TicketTypeID is unique and not empty
PASS | Dim_TicketClass.TicketClassID is unique and not empty
PASS | Dim_Railcard.RailcardID is unique and not empty
PASS | Dim_PaymentMethod.PaymentMethodID is unique and not empty
PASS | Dim_PurchaseType.PurchaseTypeID is unique and not empty
PASS | Dim_DelayReason.DelayReasonID is unique and not empty
PASS | Fact_Journey.JourneyID is unique and not empty
PASS | Fact_TicketSales.Transaction_ID is unique and not empty
PASS | Dim_Route.Departure_StationID -> Dim_Station.StationID
PASS | Dim_Route.Arrival_StationID -> Dim_Station.StationID
PASS | Fact_Journey.RouteID -> Dim_Route.RouteID
PASS | Fact_Journey.JourneyDateKey -> Dim_Date.DateKey
PASS | Fact_Journey.DelayReasonID -> Dim_DelayReason.DelayReasonID
PASS | Fact_TicketSales.JourneyID -> Fact_Journey.JourneyID
PASS | Fact_Tick

All checks pass:

- Every primary key is unique and has no empty values.
- Every foreign key value exists in the table it points to, so there are no broken relationships.
- No ticket was lost, and the total price is still £741,921.

Last, we check the empty values. The only expected ones are the actual arrival time and the delay of cancelled journeys.

In [34]:
for name, table in tables.items():
    empty = table.isna().sum()
    empty = empty[empty > 0]
    if len(empty) > 0:
        print(name, dict(empty))

cancelled = (fact_journey["Journey_Status"] == "Cancelled").sum()
print("Cancelled journeys:", cancelled)

Fact_Journey {'Actual_Arrival_Time': np.int64(790), 'Delay_Minutes': np.int64(790)}
Cancelled journeys: 790


Only `Fact_Journey` has empty values, in `Actual_Arrival_Time` and `Delay_Minutes`, and their number equals the number of cancelled journeys. All other tables are complete.

## Part E: Export

We save each table as a CSV file in `data/processed/`. The file name is the table name.

In [35]:
import os

output_dir = "../data/processed"
os.makedirs(output_dir, exist_ok=True)

for name, table in tables.items():
    table.to_csv(f"{output_dir}/{name}.csv", index=False)
    print(f"{name + '.csv':<24} {len(table):>6} rows")

Dim_Station.csv              32 rows
Dim_Route.csv                65 rows
Dim_Date.csv                152 rows
Dim_TicketType.csv            3 rows
Dim_TicketClass.csv           2 rows
Dim_Railcard.csv              4 rows
Dim_PaymentMethod.csv         3 rows
Dim_PurchaseType.csv          2 rows
Dim_DelayReason.csv           6 rows
Fact_Journey.csv          19871 rows
Fact_TicketSales.csv      31653 rows


The 11 files are saved and ready to be loaded into SQL Server.

## Summary

- The raw data was cleaned without removing any row: delay reasons were standardized, missing railcards became `None`, dates and times were converted, and two new columns were calculated (`Days_Booked_Ahead` and `Delay_Minutes`, with the midnight rule).
- `Region`, `Distance_km` and the station locations (`Latitude`, `Longitude`) were added from the ORR station attributes file. The distance is a straight-line distance.
- The data was split into 9 dimension tables and 2 fact tables that follow the project schema.
- `Fact_TicketSales` has 31,653 tickets and `Fact_Journey` has 19,871 journeys.
- All primary keys are unique, all foreign keys are valid, and the total price (£741,921) is unchanged.